In [1]:
# ============================================================
# E4 Wristband Sleep Staging
# Lightweight 1D-CNN + BiGRU  (NOT BiT-MamSleep)
#
# WHY a different (lighter) architecture than the EEG model:
#   - EEG needs multi-resolution CNN + Mamba because the signal
#     is oscillatory (delta/theta/alpha/spindle bands) at fine
#     temporal scale.
#   - E4 (BVP, HR, TEMP) is slow-varying autonomic/thermoregulatory
#     signal - no fine oscillatory structure. Using the heavy
#     EEG-style architecture here overfits / wastes capacity
#     (this is what caused the ~48% acc / F1=0.457 result earlier).
#   - So: small CNN encoder (downsamples 1920 -> ~120 timesteps)
#     + BiGRU on top for temporal context, simple classifier head.
#
# Input : E4 .npz files -> keys: e4 (N,3,1920), labels (N,)
#         channels = [BVP, HR, TEMP]  (from v3 preprocessing)
# Output: D:\22\AA\evaluation\e4_cnnbigru_BHT
# ============================================================

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, cohen_kappa_score, accuracy_score, classification_report
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# PATHS / CONFIG
# ============================================================
DATA_PATH  = r"D:\22\AA\preprocess\preprocessed_E4_v3_BHT"
EVAL_PATH  = r"D:\22\AA\evaluation\e4_cnnbigru_BHT"
os.makedirs(EVAL_PATH, exist_ok=True)

LABEL_NAMES = ["Wake", "N1", "N2", "N3", "REM"]
N_CLASSES   = 5

BATCH_SIZE   = 128
EPOCHS       = 40
PATIENCE     = 8            # early stopping on val macro-F1
LR           = 1e-3
WEIGHT_DECAY = 1e-4

# SupCon is OPTIONAL here and OFF by default per earlier analysis
# (autonomic signal separability is low; contrastive loss can
# amplify noise). Flip to True + small lambda only if you want
# to test it after the plain CNN+BiGRU baseline is established.
USE_SUPCON    = False
SUPCON_LAMBDA = 0.05
SUPCON_TEMP   = 0.07

SEEDS = [42, 7, 123, 2024, 777]   # 5-seed ensemble, same convention as EEG pipeline

VAL_SUBJECT_FRACTION = 0.15   # carved out of TRAIN subjects, subject-level split


# ============================================================
# CUDA-safe device (from your earlier fix)
# ============================================================
def get_safe_device():
    if not torch.cuda.is_available():
        print("CUDA not available, using CPU")
        return torch.device("cpu")
    try:
        torch.cuda.init()
        test = torch.zeros(1).cuda()
        _ = test + 1
        torch.cuda.synchronize()
        print(f"CUDA OK: {torch.cuda.get_device_name(0)}")
        return torch.device("cuda")
    except RuntimeError as e:
        print(f"CUDA init failed ({e}), falling back to CPU")
        return torch.device("cpu")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ============================================================
# DATASET
# ============================================================
class E4Dataset(Dataset):
    """Loads all subject .npz files fully into RAM (small: 3ch x 1920 samples)."""

    def __init__(self, subject_ids, data_path=DATA_PATH):
        self.X = []
        self.y = []
        self.subject_of_epoch = []

        for sub in subject_ids:
            fpath = os.path.join(data_path, f"{sub}.npz")
            if not os.path.exists(fpath):
                continue
            with np.load(fpath) as d:
                e4 = d['e4']          # (n, 3, 1920)
                labels = d['labels']  # (n,)
            self.X.append(e4)
            self.y.append(labels)
            self.subject_of_epoch.extend([sub] * len(labels))

        self.X = np.concatenate(self.X, axis=0).astype(np.float32)
        self.y = np.concatenate(self.y, axis=0).astype(np.int64)

        self.label_counts = np.array(
            [np.sum(self.y == c) for c in range(N_CLASSES)], dtype=np.float64
        )

        mb = self.X.nbytes / 1e9
        print(f"  Samples : {len(self.y):,}")
        print(f"  RAM     : {mb:.2f} GB")

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), torch.tensor(self.y[idx])


def subject_level_split(train_subs, val_fraction, seed=42):
    rng = random.Random(seed)
    subs = list(train_subs)
    rng.shuffle(subs)
    n_val = max(1, int(len(subs) * val_fraction))
    val_subs   = subs[:n_val]
    train_subs = subs[n_val:]
    return train_subs, val_subs


# ============================================================
# MODEL: lightweight CNN encoder + BiGRU
# ============================================================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel, stride=2, dropout=0.2):
        super().__init__()
        pad = kernel // 2
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=kernel, stride=stride, padding=pad)
        self.bn   = nn.BatchNorm1d(out_ch)
        self.act  = nn.GELU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.act(self.bn(self.conv(x))))


class E4Encoder(nn.Module):
    """1920 samples -> ~120 timesteps, 3ch -> base*4 channels."""

    def __init__(self, in_ch=3, base=32, dropout=0.2):
        super().__init__()
        self.block1 = ConvBlock(in_ch,     base,    kernel=7, stride=2, dropout=dropout)  # 1920->960
        self.block2 = ConvBlock(base,      base*2,  kernel=7, stride=2, dropout=dropout)  # 960->480
        self.block3 = ConvBlock(base*2,    base*4,  kernel=5, stride=2, dropout=dropout)  # 480->240
        self.block4 = ConvBlock(base*4,    base*4,  kernel=5, stride=2, dropout=dropout)  # 240->120
        self.out_ch = base * 4

    def forward(self, x):          # x: (B, 3, 1920)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)         # (B, base*4, ~120)
        return x


class E4CNNBiGRU(nn.Module):
    def __init__(self, in_ch=3, base=32, gru_hidden=64, n_classes=N_CLASSES,
                 dropout=0.3, return_embedding=False):
        super().__init__()
        self.encoder = E4Encoder(in_ch=in_ch, base=base, dropout=dropout)
        self.gru = nn.GRU(
            input_size=self.encoder.out_ch,
            hidden_size=gru_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        emb_dim = gru_hidden * 2
        self.emb_dim = emb_dim
        self.return_embedding = return_embedding

        self.classifier = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(emb_dim, n_classes),
        )

    def forward(self, x):                      # x: (B, 3, 1920)
        feat = self.encoder(x)                 # (B, C, T)
        feat = feat.permute(0, 2, 1)            # (B, T, C)
        gru_out, _ = self.gru(feat)             # (B, T, 2*hidden)

        avg_pool = gru_out.mean(dim=1)
        max_pool, _ = gru_out.max(dim=1)
        embedding = (avg_pool + max_pool) / 2.0  # (B, emb_dim)

        logits = self.classifier(embedding)

        if self.return_embedding:
            return logits, embedding
        return logits


# ============================================================
# SupCon loss (optional, off by default)
# ============================================================
class SupConLoss(nn.Module):
    def __init__(self, temperature=SUPCON_TEMP):
        super().__init__()
        self.temperature = temperature

    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        device = embeddings.device
        batch_size = embeddings.shape[0]

        sim = torch.matmul(embeddings, embeddings.T) / self.temperature
        sim_max, _ = sim.max(dim=1, keepdim=True)
        sim = sim - sim_max.detach()

        labels = labels.view(-1, 1)
        mask_pos = torch.eq(labels, labels.T).float().to(device)
        mask_self = torch.eye(batch_size, device=device)
        mask_pos = mask_pos - mask_self

        exp_sim = torch.exp(sim) * (1 - mask_self)
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-12)

        pos_count = mask_pos.sum(dim=1)
        pos_count = torch.clamp(pos_count, min=1.0)
        mean_log_prob_pos = (mask_pos * log_prob).sum(dim=1) / pos_count

        loss = -mean_log_prob_pos.mean()
        return loss


# ============================================================
# TRAIN / EVAL LOOPS
# ============================================================
def run_epoch(model, loader, device, class_weights, optimizer=None,
              supcon_fn=None, supcon_lambda=0.0):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds, all_labels = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        with torch.set_grad_enabled(is_train):
            if supcon_fn is not None:
                logits, emb = model(x)
                ce_loss = F.cross_entropy(logits, y, weight=class_weights)
                sc_loss = supcon_fn(emb, y)
                loss = ce_loss + supcon_lambda * sc_loss
            else:
                logits = model(x)
                loss = F.cross_entropy(logits, y, weight=class_weights)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(y.detach().cpu().numpy())

    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    avg_loss = total_loss / len(all_labels)
    acc      = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    kappa    = cohen_kappa_score(all_labels, all_preds)

    return avg_loss, acc, f1_macro, kappa, all_preds, all_labels


def train_one_seed(seed, train_subs_all, test_subs, device):
    print(f"\n{'='*60}\nSEED {seed}\n{'='*60}")
    set_seed(seed)

    tr_subs, val_subs = subject_level_split(train_subs_all, VAL_SUBJECT_FRACTION, seed=seed)

    print("Building datasets...")
    train_ds = E4Dataset(tr_subs)
    val_ds   = E4Dataset(val_subs)
    test_ds  = E4Dataset(test_subs)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    cw_np = train_ds.label_counts.sum() / (N_CLASSES * train_ds.label_counts)
    cw = torch.FloatTensor(cw_np).to(device)
    print("Class weights:")
    for name, w in zip(LABEL_NAMES, cw_np):
        print(f"  {name:6s}: {w:.3f}")

    model = E4CNNBiGRU(in_ch=3, base=32, gru_hidden=64,
                        n_classes=N_CLASSES, dropout=0.3,
                        return_embedding=USE_SUPCON).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )

    supcon_fn = SupConLoss(temperature=SUPCON_TEMP) if USE_SUPCON else None

    best_val_f1 = -1.0
    best_state  = None
    epochs_no_improve = 0

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc, tr_f1, tr_kappa, _, _ = run_epoch(
            model, train_loader, device, cw, optimizer=optimizer,
            supcon_fn=supcon_fn, supcon_lambda=SUPCON_LAMBDA
        )
        val_loss, val_acc, val_f1, val_kappa, _, _ = run_epoch(
            model, val_loader, device, cw, optimizer=None,
            supcon_fn=supcon_fn, supcon_lambda=SUPCON_LAMBDA
        )
        scheduler.step(val_f1)

        print(f"Ep{epoch:02d} | Train loss={tr_loss:.3f} acc={tr_acc:.3f} f1={tr_f1:.3f} "
              f"| Val loss={val_loss:.3f} acc={val_acc:.3f} f1={val_f1:.3f} kappa={val_kappa:.3f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
                break

    model.load_state_dict(best_state)
    ckpt_path = os.path.join(EVAL_PATH, f"e4_cnnbigru_seed{seed}.pt")
    torch.save(best_state, ckpt_path)
    print(f"Saved best checkpoint -> {ckpt_path}")

    # Final test eval for this seed
    test_loss, test_acc, test_f1, test_kappa, test_preds, test_labels = run_epoch(
        model, test_loader, device, cw, optimizer=None,
        supcon_fn=supcon_fn, supcon_lambda=SUPCON_LAMBDA
    )
    print(f"\n[Seed {seed}] TEST: acc={test_acc:.4f} f1={test_f1:.4f} kappa={test_kappa:.4f}")
    print(classification_report(test_labels, test_preds, target_names=LABEL_NAMES, digits=4))

    # Also collect softmax probs on test set for later ensembling
    model.eval()
    all_probs = []
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            if USE_SUPCON:
                logits, _ = model(x)
            else:
                logits = model(x)
            probs = F.softmax(logits, dim=1)
            all_probs.append(probs.cpu().numpy())
    all_probs = np.concatenate(all_probs, axis=0)

    return {
        'seed': seed,
        'test_acc': test_acc,
        'test_f1': test_f1,
        'test_kappa': test_kappa,
        'test_probs': all_probs,
        'test_labels': test_labels,
    }


def ensemble_results(results):
    avg_probs = np.mean([r['test_probs'] for r in results], axis=0)
    labels = results[0]['test_labels']
    preds = avg_probs.argmax(axis=1)

    acc   = accuracy_score(labels, preds)
    f1    = f1_score(labels, preds, average='macro')
    kappa = cohen_kappa_score(labels, preds)

    print(f"\n{'='*60}\nENSEMBLE ({len(results)} seeds)\n{'='*60}")
    print(f"Acc={acc:.4f}  F1={f1:.4f}  Kappa={kappa:.4f}")
    print(classification_report(labels, preds, target_names=LABEL_NAMES, digits=4))

    np.savez(
        os.path.join(EVAL_PATH, "ensemble_results.npz"),
        avg_probs=avg_probs, labels=labels, preds=preds,
        acc=acc, f1=f1, kappa=kappa
    )
    return acc, f1, kappa


# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    device = get_safe_device()

    train_subs_all = np.load(os.path.join(DATA_PATH, "_train_subs.npy"), allow_pickle=True).tolist()
    test_subs      = np.load(os.path.join(DATA_PATH, "_test_subs.npy"),  allow_pickle=True).tolist()

    print(f"Device      : {device}")
    print(f"Architecture: Lightweight CNN(in_ch=3) + BiGRU")
    print(f"E4 channels : BVP, HR, TEMP")
    print(f"Output      : {EVAL_PATH}")
    print(f"Train subs (pool for train/val split): {len(train_subs_all)}  Test: {len(test_subs)}")
    print(f"SupCon      : {'ON (lambda=' + str(SUPCON_LAMBDA) + ')' if USE_SUPCON else 'OFF'}")

    all_results = []
    for seed in SEEDS:
        result = train_one_seed(seed, train_subs_all, test_subs, device)
        all_results.append(result)

    print(f"\n{'='*60}\nPER-SEED SUMMARY\n{'='*60}")
    for r in all_results:
        print(f"  Seed {r['seed']:>5}: Acc={r['test_acc']:.4f}  F1={r['test_f1']:.4f}  Kappa={r['test_kappa']:.4f}")

    ensemble_results(all_results)

CUDA OK: NVIDIA GeForce RTX 4090
Device      : cuda
Architecture: Lightweight CNN(in_ch=3) + BiGRU
E4 channels : BVP, HR, TEMP
Output      : D:\22\AA\evaluation\e4_cnnbigru_BHT
Train subs (pool for train/val split): 73  Test: 19
SupCon      : OFF

SEED 42
Building datasets...
  Samples : 58,830
  RAM     : 1.36 GB
  Samples : 10,027
  RAM     : 0.23 GB
  Samples : 18,899
  RAM     : 0.44 GB
Class weights:
  Wake  : 1.842
  N1    : 3.124
  N2    : 0.446
  N3    : 0.987
  REM   : 1.137
Ep01 | Train loss=1.428 acc=0.385 f1=0.345 | Val loss=1.478 acc=0.337 f1=0.312 kappa=0.112
Ep02 | Train loss=1.392 acc=0.391 f1=0.361 | Val loss=1.506 acc=0.318 f1=0.299 kappa=0.118
Ep03 | Train loss=1.371 acc=0.395 f1=0.368 | Val loss=1.521 acc=0.325 f1=0.302 kappa=0.118
Ep04 | Train loss=1.351 acc=0.400 f1=0.376 | Val loss=1.559 acc=0.333 f1=0.298 kappa=0.124
Ep05 | Train loss=1.335 acc=0.405 f1=0.381 | Val loss=1.656 acc=0.318 f1=0.288 kappa=0.118
Ep06 | Train loss=1.302 acc=0.422 f1=0.399 | Val loss=1.

KeyboardInterrupt: 